# Optimizer

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display

$$
f(x_1, x_2) = (x_1 - 2)^2 + (x_2 - 3)^2
$$

In [ ]:
def f_(x, y):
    return (x - 2) ** 2 + 5 * (y - 3) ** 2

d = torch.tensor([2., 3.])
def f(x):
    return (x[0] - 2).pow(2) + 5 * (x[1] - 3).pow(2)

x = np.linspace(-10, 10, 200)
y = np.linspace(-10, 10, 200)
X, Y = np.meshgrid(x, y)
X = torch.tensor(X)
Y = torch.tensor(Y)
Z = f_(X, Y)

plt.figure(figsize=(7, 5))
plt.contour(X, Y, Z, levels=10, cmap="gist_gray")
extent = [x.min(), x.max(), y.min(), y.max()]
plt.imshow(
    Z, extent=extent,
    origin='lower', cmap='viridis'
)
plt.show()

In [ ]:
def train(optimizer, n=30):
    history = [optimizer.params]
    for _ in range(n):
        y = f(optimizer.params)
        y.backward()
        optimizer.step()
        history.append(optimizer.params)

    fig, ax = plt.subplots(figsize=(7, 5))
    plt.contour(X, Y, Z, levels=10, cmap="gist_gray")
    ax.imshow(Z, extent=extent, origin='lower', cmap='viridis')

    line, = ax.plot([], [], marker='o', color='red', markersize=3)

    pts = [p.detach().numpy() for p in history]
    xs, ys = zip(*pts)

    def update(i):
        line.set_data(xs[:i+1], ys[:i+1])
        return line,

    ani = animation.FuncAnimation(
        fig, update, frames=len(xs),
        interval=20, blit=True, repeat=False
    )
    display(HTML(ani.to_jshtml()))
    plt.close()

---

## SGD

In [ ]:
class SGD:
    def __init__(self, params, lr=0.01):
        self.params = params
        self.lr = lr

    def step(self):
        self.params = self.params - self.lr * self.params.grad
        self.params.detach_().requires_grad_()

In [ ]:
x = torch.tensor([-7.0, -8.0], requires_grad=True)
optimizer = SGD(params=x, lr=0.05)
train(optimizer, n=100)

## モーメンタム

慣性の利用

In [ ]:
class MomentumSGD:
    def __init__(self, params, lr=0.01, beta=0.9):
        self.params = params
        self.lr = lr
        self.beta = beta
        self.v = None

    def step(self):
        grad = self.params.grad
        if self.v is None:
            self.v = grad
        else:
            self.v = self.beta * self.v + (1 - self.beta) * grad
        self.params = self.params - self.lr * self.v
        self.params.detach_().requires_grad_()

In [ ]:
x = torch.tensor([-7.0, -8.0], requires_grad=True)
optimizer = MomentumSGD(params=x, lr=0.01, beta=0.9)
train(optimizer, n=100)

## Nesterov

慣性方向に少し先を見て勾配を計算する。

In [ ]:
class NesterovMomentumSGD:
    def __init__(self, params, lr=0.01, beta=0.9):
        self.params = params
        self.lr = lr
        self.beta = beta
        self.v = None

    def step(self):
        grad = self.params.grad
        if self.v is None:
            self.v = grad
        else:
            self.v = self.beta * self.v + (1 - self.beta) * grad
        self.params = self.params - self.lr * (grad + self.beta * self.v)
        self.params.detach_().requires_grad_()

In [ ]:
x = torch.tensor([-7.0, -8.0], requires_grad=True)
optimizer = NesterovMomentumSGD(params=x, lr=0.01, beta=0.9)
train(optimizer, n=100)

理論通りのバージョン

In [ ]:
def train_nest(optimizer, n=30):
    history = [optimizer.params]
    for _ in range(n):
        y = f(optimizer.running_params)
        y.backward()
        optimizer.step()
        history.append(optimizer.params)

    fig, ax = plt.subplots(figsize=(7, 5))
    plt.contour(X, Y, Z, levels=10, cmap="gist_gray")
    ax.imshow(Z, extent=extent, origin='lower', cmap='viridis')

    line, = ax.plot([], [], marker='o', color='red', markersize=3)

    pts = [p.detach().numpy() for p in history]
    xs, ys = zip(*pts)

    def update(i):
        line.set_data(xs[:i+1], ys[:i+1])
        return line,

    ani = animation.FuncAnimation(
        fig, update, frames=len(xs),
        interval=20, blit=True, repeat=False
    )
    display(HTML(ani.to_jshtml()))
    plt.close()

class NesterovMomentumSGD:
    def __init__(self, params, lr=0.01, beta=0.9):
        self.params = params
        self.running_params = params
        self.lr = lr
        self.beta = beta
        self.v = None

    def step(self):
        grad = self.running_params.grad
        if self.v is None:
            self.v = self.lr * grad
        else:
            self.v = self.beta * self.v + self.lr * grad
        self.params = self.params - self.v
        self.running_params = self.params - self.beta * self.v
        self.params.detach_().requires_grad_()
        self.running_params.detach_().requires_grad_()

x = torch.tensor([-7.0, -8.0], requires_grad=True)
optimizer = NesterovMomentumSGD(params=x, lr=0.01, beta=0.9)
train_nest(optimizer, n=100)

## RMSProp

絶対値が大きすぎる方向の勾配を抑える。絶対値の大きさを表すRMS(二乗平均平方根)で割るだけ。振動の抑制に繋がる。

勾配が大きな方向には大きくステップすべきである、という仮説からは外れることになるが、気にしているのは現在の勾配ではなく最近の勾配なのでそんなに問題ないというか。振動せずにめちゃくちゃ大きな坂を下りまくっているのであれば話は別だが、そうでもない限り一方向の最近の勾配の大きさが増大することはなく、実際にそんな場面ほとんどないから無視していいみたいなイメージかな。

In [ ]:
class RMSProp:
    def __init__(self, params, lr=0.01, beta=0.9):
        self.params = params
        self.lr = lr
        self.beta = beta
        self.s = None

    def step(self):
        grad = self.params.grad
        if self.s is None:
            self.s = torch.zeros_like(grad)
        self.s = self.beta * self.s + (1 - self.beta) * grad ** 2
        self.params = self.params - self.lr * grad / torch.sqrt(self.s + 1e-8)
        self.params.detach_().requires_grad_()

In [ ]:
x = torch.tensor([-7.0, -8.0], requires_grad=True)
optimizer = RMSProp(params=x, lr=0.2, beta=0.9)
train(optimizer, n=100)